two things
# 1. spark optimization techniques
# 2. delta optimization techniques

In [0]:
from pyspark.sql.functions import col
df= spark.table("formula1_dev.bronze.circuits")
df1=df.filter(col("circuit_id")==71)\
    .select(col("circuit_ref"),col("circuit_id"))\
    .orderBy(col("circuit_id"))

In [0]:
# df1.explain("formatted")
df1.explain()

In [0]:
from 
join
filter
group 
having
select 
order 
limit 

In [0]:
predicate pushdown ==> early filter 
column pruning ==> select specific columns


In [0]:
shuffle --> data across the partitions [spark]
to reduce --> partitions of the data should be reduced. 
to read the files --> process the data .. shuffle will be more 
low cardinality columns --> reduce the shuffle --> country , location ,region 
# partitionBy("country")

In [0]:
executors --> work chesthundi --> plan -->transformations
driver --> results 

In [0]:
DAG --> direct/directed acyclic graph 
when you code --> run
directed --> every edge has a direction data flow from one operation to another operation [parent --> child ]
-->*-->*
Acyclic --> no cycles --> no loops --> no circular dependencies 
Graph --> collection of nodes and eges 

In [0]:
spark builds the dag lazily --> when you run the code like filter ,joins , groupby
spark dag lazy build execution bluepront for your job 

In [0]:
transformations -->  narrow and wide transformations 
every transformation filter, group by --> just adds a node to the logical dag 
spark tracks two kinds of dependency between nodes:
    1. Narrow dependency/narrow transformation --> parent and child partitions are in the same executor
    parent--> child partitions are in the same executor 
    2.wide dependency/wide transformation --> parent and child partitions are in different executors
    parent -> multiple child partitions 
1 partition --> multiple partitions create avuthundi

In [0]:
print(spark.version)

In [0]:
3. things for aqe
1. spark.sql.shuffle.partitions = 200[default] this creates numbr of partitions at the time of the shuffling 
spark.sql.maxPartitionBytes = 128 MB this is the max size of the partition

In [0]:
spark.conf.set("spark.sql.maxPartitionBytes", 512) # file reading partition size 
spark.conf.set("spark.sql.shuffle.partitions", 250) # shuffling the data across the executors transfomations like joins, group by 

In [0]:
df.write.mode("delta").partitionBy("state").saveAsTable("table_name")



In [0]:
10 MB --> partition size of the data in the memory
10 MB --> 10MB 

In [0]:
results_df = spark.table("formula1_dev.silver.results")
races_df = spark.table("formula1_dev.silver.races")
drivers_df = spark.table("formula1_dev.silver.drivers")



demo_races_df = races_df
demo_results_df = results_df

print("Demo races:", demo_races_df.count())
print("Demo results:", demo_results_df.count())

In [0]:
baseline_df = (
    demo_results_df.alias("r")
    .join(
        demo_races_df.alias("ra"),
        F.col("r.race_id") == F.col("ra.race_id"),
        "inner"
    )
    .join(
        drivers_df.alias("d"),
        F.col("r.driver_id") == F.col("d.driver_id"),
        "inner"
    )
    .groupBy(
        "race_year",
        F.col("r.driver_id"),
        F.col("d.driver_full_name")
    )
    .agg(
        F.sum("r.points").alias("total_points"),
        F.countDistinct("r.race_id").alias("races_entered")
    )
)

baseline_df.orderBy(
    "race_year",
    F.col("total_points").desc()
).show(20, False)

In [0]:
from pyspark.sql.functions import *
shuffle_df = (
    demo_results_df
    .groupBy("driver_id")
    .agg(sum("points").alias("total_points"))
)

shuffle_df.show(10, False)
shuffle_df.explain(True)

In [0]:
# results_df.count() 25000
# races_df.count() 1058

In [0]:
# 2. broadcast join
from pyspark.sql.functions import * 
broadcast_join_df = demo_results_df.alias("r").join(
    broadcast(demo_races_df).alias("ra"),
    col("r.race_id") == col("ra.race_id"), "inner")
broadcast_join_df.count()

In [0]:
broadcast_join_df.explain()

In [0]:
# 2. broadcast join
from pyspark.sql.functions import * 
broadcast_join_df1 = demo_results_df.alias("r").join(
    demo_races_df.alias("ra"),
    col("r.race_id") == col("ra.race_id"), "inner")
broadcast_join_df1.count()

In [0]:
broadcast_join_df1.explain()

In [0]:
# improper data distribution across partitions - skew data
# shuffle is data distrubtion across pratitions 

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1) # no broad cast join threshold

In [0]:
drivers_df.count()

In [0]:
#sort merge join 

results_df.alias("r").join(drivers_df.alias("d"), col("r.driver_id") == col("d.driver_id"), "inner").explain()

In [0]:
shuffle_hash_df1 = results_df.alias("r").join(drivers_df.alias("d"), col("r.driver_id") == col("d.driver_id"), "inner").hint("shuffle_hash")
shuffle_hash_df1.explain("formatted")

In [0]:
external and internal tables 
path == > external 
managed/internal table ---> no path 

In [0]:
cache /persist / unpersist 
coalesce / repartion
vaccum 


In [0]:
df.write.mode("overwrite").option(mergeSchema, "True").saveAsTable("table_name") # schema evolution # schema enforcement
df.write.mode("overwrite").format("delta").saveAsTable("table_name")  # schema enforcement 

In [0]:
complete stage will take time to complete --> data skew
uneven distribution of the data across partitions ---> skew data

In [0]:
#salting technique 
from pyspark.sql.functions import col
skew_df = results_df.alias("r1").join(drivers_df.alias("r2"), col("r1.driver_id") == col("r2.driver_id"), "inner")
skew_df.explain("formatted")

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import pmod
salt_count = 10
salt_df = skew_df.withColumn("salt", pmod(f.rand()*salt_count, salt_count))
salt_df.display()

In [0]:
#salting splits a hot key/skewed key into multiple articifical key. This will distribute the data evenly across the partitions

In [0]:
1.AQE
2.repartition 
3.salting 

In [0]:
shuffle partitions =200 
spark.conf.set(spark.sql.maxPartitionBytes, 128MB)
spark.conf.set("spark.sql.shuffle.partitions", "200")

In [0]:
df.repartition(8) # it will increase partitions so that data will be evenly distributed across the partitions. shuffle the data but it will not change the data.
df.coalesce(2) # it will reduce the partitions no shuffle 
# to reduce the files while writing
df.coalesce(2).write.mode("overwrite").format("delta").saveAsTable("table_name")

In [0]:
df.cache() # to store the intermediate results in the executor memory[RAM]
df1.cache()
df.persist() # to store the intermediate results but you store in the disk & RAM
df.persist(storageLevel.MEMORY_AND_DISK)
df.unpersist() # REMOVE TO THE CACHE/PERSIST data from the executor memory

In [0]:
df.unpersist()
df1.unpersist()

In [0]:
silver_table = "formula1_dev.silver.results"
try:
    df= spark.sql(f"describe detail {silver_table}")
    # df.select("format", "numFiles","sizeInBytes").display()
    df.display()
except Exception as e:
    print(e)
    print("Table does not exists")

In [0]:
spark.sql(f"optimize formula1_dev.silver.results") 
 # small files into large files

In [0]:
%sql
desc history formula1_dev.silver.results

In [0]:
# zorder -->max and min[
#     #frequently filtering columns 


In [0]:
spark.sql("optimize formula1_dev.silver.results zorder by race_id")

In [0]:
# slow jobs /long running jobs 

In [0]:
merge/insert --> DML
merge --> keys [andhulo data duplicate]
source side 

In [0]:
%sql
show create table formula1_dev.silver.races

In [0]:
df.explain()
#AQE --> 
#OPTIMIZE --> SMALL FILES INTO THE LARGE / TARGET FILE SIZE = 128 mb/256 mb 
 'delta.feature.deletionVectors' = 'supported', --> it will mark internally as deleted and will not delete the data physically
 partition should be done on low cardinality like region ,country
 df.partitionBy("region") --> it will create the partition folder
 optimize write --> optimize the data while creating inot the delta table 
 liquid clustering -- > optimize +partitions+zorder 

In [0]:
%sql
alter table formula1_dev.silver.races
set tblproperties (delta.autoOptimize.optimizeWrite = true, delta.autoOptimize.autoCompact = true)

In [0]:
%sql
alter table formula1_dev.silver.races
cluster by auto

In [0]:
%sql
desc detail formula1_dev.silver.races